# Automobile Sales During Recession Periods
## Exploratory Data Analysis & Visualization Report

**Focus:** historical automobile sales, recession effects, advertising, macroeconomic conditions, seasonality, and vehicle-type performance  
**Tools:** Python · pandas · NumPy · Matplotlib · Seaborn

---

### Executive Summary

This analysis examines how automobile sales behaved across recession and non-recession periods from 1980 to 2023. The strongest recurring pattern is a broad contraction in sales during recessions, although the magnitude differs by vehicle type. Economic conditions also matter: higher unemployment and higher vehicle prices generally coincide with weaker sales, while consumer confidence shows a weaker and noisier positive relationship.

Advertising expenditure is substantially concentrated in non-recession periods, and its movement does not consistently track automobile sales year to year. Seasonality introduces additional variation, with some months showing considerably higher sales activity than others. Together, the results suggest that automobile demand is shaped by a combination of macroeconomic pressure, affordability, vehicle segment, and seasonal timing rather than any single factor.

## 1. Project Questions

The report addresses six practical questions:

1. How have average automobile sales changed over time?
2. How different are sales during recession and non-recession periods, and which vehicle types are most affected?
3. Do advertising expenditure and automobile sales move together during non-recession periods?
4. How do GDP, unemployment, consumer confidence, and vehicle price relate to sales?
5. What seasonal patterns appear in automobile demand?
6. How is advertising expenditure distributed across economic conditions and vehicle types?

### Data Source

The analysis uses the **Automobile Sales** dataset provided through IBM Skills Network coursework. The dataset contains monthly observations from 1980–2023 and includes sales, economic indicators, advertising expenditure, vehicle type, seasonality, and recession status.

## 2. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

### 2.1 Load and prepare the data

In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "d51iMGfp_t0QpO30Lym-dw/automobile-sales.csv"
)

local_file = Path("automobile-sales.csv")
df = pd.read_csv(local_file if local_file.exists() else DATA_URL)

df["Date"] = pd.to_datetime(df["Date"])
df["Recession_Label"] = df["Recession"].map({0: "Non-Recession", 1: "Recession"})

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

df.head()

### 2.2 Data quality check

In [ ]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Date range",
        "Missing values",
        "Duplicate rows",
        "Recession observations"
    ],
    "Value": [
        f"{len(df):,}",
        df.shape[1],
        f"{df['Date'].min().date()} to {df['Date'].max().date()}",
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        f"{(df['Recession'] == 1).sum():,} ({(df['Recession'].mean() * 100):.1f}%)"
    ]
})

display(quality_summary)

The dataset spans multiple economic cycles, allowing recession and non-recession observations to be compared directly. Before interpreting the visualizations, the analysis checks record counts, missing values, duplicate rows, and the balance of recession observations to make the workflow reproducible and transparent.

## 3. Sales Trend Over Time

In [ ]:
yearly_sales = (
    df.groupby("Year", as_index=False)["Automobile_Sales"]
      .mean()
      .rename(columns={"Automobile_Sales": "Average_Sales"})
)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.lineplot(data=yearly_sales, x="Year", y="Average_Sales",
             marker="o", linewidth=2, ax=ax)

for start, end in [(1980, 1982), (1991, 1991), (2000, 2001), (2007, 2009), (2020, 2020)]:
    ax.axvspan(start, end if end > start else start + 0.35, alpha=0.10)

ax.set(
    title="Average Automobile Sales by Year",
    xlabel="Year",
    ylabel="Average automobile sales"
)
ax.text(
    0.01, -0.20,
    "Shaded bands indicate recession years represented in the dataset.",
    transform=ax.transAxes, fontsize=9
)
plt.tight_layout()
plt.show()

**Interpretation.** Automobile sales are highly variable across the period, with several pronounced peaks and contractions rather than a smooth long-run trend. Many of the weakest periods overlap with recession years, supporting the expectation that vehicle demand is sensitive to economic downturns. The large year-to-year swings also indicate that recession status alone does not explain all variation in sales.

## 4. Recession Effect by Vehicle Type

In [ ]:
sales_by_vehicle = (
    df.groupby(["Recession_Label", "Vehicle_Type"], as_index=False)["Automobile_Sales"]
      .mean()
)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(
    data=sales_by_vehicle,
    x="Recession_Label",
    y="Automobile_Sales",
    hue="Vehicle_Type",
    order=["Non-Recession", "Recession"],
    ax=ax
)
ax.set(
    title="Average Sales by Vehicle Type and Economic Period",
    xlabel="Economic period",
    ylabel="Average automobile sales"
)
ax.legend(title="Vehicle type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
vehicle_impact = (
    sales_by_vehicle
    .pivot(index="Vehicle_Type", columns="Recession_Label", values="Automobile_Sales")
    .assign(
        Percent_Change=lambda x:
        (x["Recession"] - x["Non-Recession"]) / x["Non-Recession"] * 100
    )
    .sort_values("Percent_Change")
)

display(vehicle_impact.round(2))

**Interpretation.** Every vehicle category records lower average sales during recession periods. The size of the decline differs by segment, showing that recession exposure is not uniform across the product mix. This makes vehicle type an important dimension when evaluating demand risk rather than relying only on company-wide averages.

## 5. Advertising Expenditure and Sales

In [ ]:
non_recession = df[df["Recession"] == 0].copy()

yearly_nonrec = (
    non_recession.groupby("Year", as_index=False)
    .agg(
        Automobile_Sales=("Automobile_Sales", "mean"),
        Advertising_Expenditure=("Advertising_Expenditure", "mean")
    )
)

indexed = yearly_nonrec.copy()
for col in ["Automobile_Sales", "Advertising_Expenditure"]:
    indexed[f"{col}_Index"] = indexed[col] / indexed[col].iloc[0] * 100

corr_adv_sales = yearly_nonrec["Automobile_Sales"].corr(
    yearly_nonrec["Advertising_Expenditure"]
)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.lineplot(data=indexed, x="Year", y="Automobile_Sales_Index",
             marker="o", label="Sales index", ax=ax)
sns.lineplot(data=indexed, x="Year", y="Advertising_Expenditure_Index",
             marker="s", label="Advertising index", ax=ax)
ax.axhline(100, linestyle="--", linewidth=1, alpha=0.6)
ax.set(
    title="Sales vs. Advertising During Non-Recession Periods",
    xlabel="Year",
    ylabel="Index (first observed year = 100)"
)
ax.text(
    0.99, 0.03,
    f"Pearson correlation = {corr_adv_sales:.2f}",
    transform=ax.transAxes, ha="right", va="bottom"
)
plt.tight_layout()
plt.show()

**Interpretation.** Advertising expenditure and automobile sales sometimes move in the same direction, but the relationship is inconsistent across years. Several sales peaks occur without matching advertising peaks, suggesting that advertising is only one of several forces associated with demand. The correlation shown above summarizes association, not causation.

## 6. GDP Across Economic Periods

In [ ]:
gdp_by_year = (
    df.groupby(["Year", "Recession_Label"], as_index=False)["GDP"]
      .mean()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, period in zip(axes, ["Non-Recession", "Recession"]):
    subset = gdp_by_year[gdp_by_year["Recession_Label"] == period]
    sns.lineplot(data=subset, x="Year", y="GDP", marker="o", ax=ax)
    ax.set_title(period)
    ax.set_xlabel("Year")
    ax.set_ylabel("Average GDP" if period == "Non-Recession" else "")

fig.suptitle("GDP During Recession and Non-Recession Periods", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation.** GDP is generally weaker and more compressed during recession observations than during non-recession periods. The recession series also highlights how economic contractions appear as distinct episodes rather than a continuous trend. This broader macroeconomic context helps explain why vehicle demand falls during downturns.

## 7. Seasonality in Automobile Sales

In [ ]:
monthly_nonrec = (
    non_recession.groupby("Month", observed=False, as_index=False)
    .agg(
        Automobile_Sales=("Automobile_Sales", "mean"),
        Seasonality_Weight=("Seasonality_Weight", "mean")
    )
)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.scatterplot(
    data=monthly_nonrec,
    x="Month",
    y="Automobile_Sales",
    size="Seasonality_Weight",
    sizes=(120, 900),
    hue="Seasonality_Weight",
    palette="viridis",
    ax=ax
)
ax.set(
    title="Seasonality and Average Automobile Sales",
    xlabel="Month",
    ylabel="Average automobile sales"
)
ax.legend(title="Seasonality weight", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation.** Sales vary materially by month even outside recessions, confirming that seasonality is a meaningful source of demand variation. Months with larger seasonality weights tend to be associated with higher sales activity, although the relationship is not perfectly proportional. Planning decisions should therefore distinguish recurring seasonal effects from broader economic effects.

## 8. Consumer Confidence, Price, and Sales During Recessions

In [ ]:
recession = df[df["Recession"] == 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.regplot(
    data=recession,
    x="Consumer_Confidence",
    y="Automobile_Sales",
    scatter_kws={"alpha": 0.45},
    line_kws={"linewidth": 2},
    ax=axes[0]
)
axes[0].set_title("Consumer Confidence vs. Sales")
axes[0].set_xlabel("Consumer confidence")
axes[0].set_ylabel("Automobile sales")

sns.regplot(
    data=recession,
    x="Price",
    y="Automobile_Sales",
    scatter_kws={"alpha": 0.45},
    line_kws={"linewidth": 2},
    ax=axes[1]
)
axes[1].set_title("Vehicle Price vs. Sales")
axes[1].set_xlabel("Vehicle price")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
recession_correlations = (
    recession[["Consumer_Confidence", "Price", "Automobile_Sales"]]
    .corr()["Automobile_Sales"]
    .drop("Automobile_Sales")
    .rename("Correlation with Automobile Sales")
    .to_frame()
)

display(recession_correlations.round(3))

**Interpretation.** During recessions, consumer confidence shows a positive but noisy relationship with automobile sales: stronger sentiment is generally associated with greater purchasing activity, but substantial dispersion remains. Vehicle price shows a clearer negative pattern, with higher-priced observations tending to record lower sales. These relationships are associative and should not be treated as causal effects.

## 9. Advertising Allocation

In [ ]:
ad_by_period = (
    df.groupby("Recession_Label")["Advertising_Expenditure"]
      .sum()
      .reindex(["Non-Recession", "Recession"])
)

fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(ad_by_period.index, ad_by_period.values)
ax.set(
    title="Total Advertising Expenditure by Economic Period",
    xlabel="Economic period",
    ylabel="Advertising expenditure"
)
ax.bar_label(bars, labels=[f"{v:,.0f}" for v in ad_by_period.values], padding=3)
plt.tight_layout()
plt.show()

In [ ]:
recession_ad_by_vehicle = (
    recession.groupby("Vehicle_Type", as_index=False)["Advertising_Expenditure"]
    .sum()
    .sort_values("Advertising_Expenditure", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=recession_ad_by_vehicle,
    x="Advertising_Expenditure",
    y="Vehicle_Type",
    ax=ax
)
ax.set(
    title="Advertising Expenditure by Vehicle Type During Recessions",
    xlabel="Total advertising expenditure",
    ylabel="Vehicle type"
)
plt.tight_layout()
plt.show()

**Interpretation.** Most total advertising expenditure occurs during non-recession periods, consistent with the larger number of normal-period observations and stronger market conditions. Within recession periods, advertising is distributed across all vehicle categories, with some segments receiving a noticeably larger share than others. Total expenditure should be interpreted alongside the number of observations in each period; it is not a direct measure of advertising intensity.

## 10. Unemployment and Vehicle Sales During Recessions

In [ ]:
unemployment_sales = (
    recession.groupby(["unemployment_rate", "Vehicle_Type"], as_index=False)
    ["Automobile_Sales"].mean()
)

fig, ax = plt.subplots(figsize=(11, 5.8))
sns.lineplot(
    data=unemployment_sales,
    x="unemployment_rate",
    y="Automobile_Sales",
    hue="Vehicle_Type",
    marker="o",
    ax=ax
)
ax.set(
    title="Unemployment Rate and Sales by Vehicle Type",
    xlabel="Unemployment rate",
    ylabel="Average automobile sales"
)
ax.legend(title="Vehicle type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation.** Sales generally weaken as unemployment rises, although the pattern is uneven across vehicle categories and includes several sharp spikes. Smaller and family-oriented vehicles remain active in parts of the recession sample, but no category is fully insulated from worsening labor-market conditions. The volatility also cautions against interpreting individual peaks as stable trends.

## 11. Key Findings

- **Recessions coincide with broad sales declines.** Every vehicle category shows lower average sales during recession periods, though the magnitude varies by segment.
- **Macroeconomic pressure matters.** Lower GDP and higher unemployment are associated with weaker automobile demand during downturns.
- **Affordability appears important.** Higher vehicle prices generally correspond to lower recession-period sales.
- **Consumer confidence helps, but does not explain sales on its own.** The positive relationship is visible but noisy.
- **Advertising and sales do not move together consistently.** This suggests that sales variation reflects a wider mix of economic and market factors.
- **Seasonality is a recurring source of demand variation.** Monthly effects should be separated from recession-driven changes when planning or forecasting.

## 12. Limitations

This is an exploratory analysis of observational data. The visualizations identify patterns and associations but do not establish causality. Several variables may be correlated with one another, and the dataset does not include every factor that can influence vehicle demand. In addition, totals such as advertising expenditure are affected by the number of observations in each economic period; normalized or per-period measures are more appropriate when comparing intensity.

The original dataset also includes a `City` field, but no geographic coordinates and no validated city-to-state mapping. A geographic sales map is therefore excluded from the core report to avoid implying spatial precision that the available fields do not support.

---

### Reproducibility

Run the notebook from top to bottom in a Python environment with `pandas`, `numpy`, `matplotlib`, and `seaborn` installed. If `automobile-sales.csv` is present in the working directory it will be used; otherwise the notebook loads the public dataset from its source URL.